<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DROID Multi-View 3D Tracking Pipeline

This notebook is a **thin orchestration layer** — all algorithm code lives in the GitHub repo.

Each stage has **two paths**:
- **🚧 Compute**: Run from scratch (for debugging a single episode)
- **☁️ Load**: Skip computation, load pre-computed results from GCS (for debugging later stages)

| Stage | Compute | Load from GCS | Output |
|---|---|---|---|
| 1. Depth | `compute_depth.py` | `gs://dm-tapnet/mv-tap/droid/depth/` | Stereo depth + gripper refinement |
| 2. Extrinsics | `compute_extrinsics.py` | `gs://dm-tapnet/mv-tap/droid/extrinsics/` | Camera-robot alignment |
| 3. Tracks | `compute_tracks.py` | `gs://dm-tapnet/mv-tap/droid/tracks/` | Dense 3D point tracks |

---
## 0. Environment Setup

In [ ]:
# @title 0a. Clone repo & install dependencies
import os

REPO_DIR = "/content/droid"
if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules https://github.com/yangyi02/droid.git {REPO_DIR}
else:
    print(f"⏭️ Repo already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull && git submodule update --init --recursive

%cd {REPO_DIR}
!bash setup.sh

In [ ]:
# @title 0b. Python imports & sys.path setup
import sys, os, json, random
import numpy as np
import torch
import mediapy as media

REPO_DIR = "/content/droid"
for p in [
    REPO_DIR,
    os.path.join(REPO_DIR, "third_party/s2m2/src"),
    os.path.join(REPO_DIR, "third_party/co-tracker"),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(REPO_DIR)
os.environ['PYOPENGL_PLATFORM'] = 'egl'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

In [ ]:
# @title 🔄 Dev: Sync from GitHub + Hot Reload (run after pushing changes)
import importlib, subprocess

result = subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())

import core.geometry, core.io, core.depth, core.physics, core.tracking
for mod in [core.geometry, core.io, core.depth, core.physics, core.tracking]:
    importlib.reload(mod)

import compute_depth, compute_extrinsics, compute_tracks, compute_2d_tracks
for mod in [compute_depth, compute_extrinsics, compute_tracks, compute_2d_tracks]:
    importlib.reload(mod)

import utils.visualization
importlib.reload(utils.visualization)
print("✅ All modules reloaded. Re-run cells below to test changes.")

In [ ]:
# @title 0c. Download DROID metadata (shared by all stages)

root_path = "/content/droid_raw/1.0.1"
base_url = "https://huggingface.co/KarlP/droid/resolve/main"
files = ["camera_serials.json", "episode_id_to_path.json",
         "keep_ranges_1_0_1.json", "cam2base_extrinsic_superset.json"]

os.makedirs(root_path, exist_ok=True)
for f in files:
    os.system(f"wget -q -nc -P {root_path} {base_url}/{f}")

def load_json(name):
    with open(os.path.join(root_path, name)) as f:
        return json.load(f)

serials_db = load_json(files[0])
id_to_path = load_json(files[1])
keep_ranges = load_json(files[2])
extrinsics_db = load_json(files[3])

# Option 1: From metadata intersection (uncomment)
# valid_ids = sorted(set(serials_db.keys()) & set(id_to_path.keys()) & set(extrinsics_db.keys()))

# Option 2: From episodes_success.txt
with open("episodes_success.txt") as f:
    valid_ids = sorted([line.strip() for line in f if line.strip()])

print(f"✅ Metadata ready: {len(valid_ids)} episodes")

In [ ]:
# @title 0d. Select episode

# Option 1: Random
episode_id = random.choice(valid_ids)

# Option 2: Manual override (uncomment)
# episode_id = "ILIAD+5e938e3b+2023-07-20-11h-50m-51s"

print(f"🎯 Episode: {episode_id}")

In [ ]:
# @title 0e. Initialize scene_constants (lightweight, no SVO)
from compute_depth import init_episode

scene_constants = init_episode(
    episode_id,
    os.path.expanduser("~/droid_data/input/robotics/droid_raw/1.0.1"),
    id_to_path, serials_db, keep_ranges)
print(f"✅ scene_constants initialized: {list(scene_constants['camera'].keys())}")

---
## 1. Stage 1: Depth

⚠️ **Run ONE of the two cells below** (Compute OR Load).

In [ ]:
# @title 1A. 🚧 COMPUTE depth from scratch (SVO decode + S2M2 + SAM)
# This cell installs ZED SDK + runs the full depth pipeline.
# Skip this entirely if using 1B (Load from GCS).

# --- Install ZED SDK (only runs once) ---
import shutil
if not shutil.which("ZED_Explorer"):
    !apt-get update -qq
    !apt-get install -y zstd
    sdk_installer = "ZED_SDK_Linux_Ubuntu22.run"
    !wget -q -O {sdk_installer} https://download.stereolabs.com/zedsdk/5.2/cu12/ubuntu22
    !chmod +x {sdk_installer}
    !./{sdk_installer} silent runtime_only skip_tools
    !find /usr/local/zed/ -name "pyzed*.whl" -exec pip install {} \;
    print("✅ ZED SDK installed")
else:
    print("⏭️ ZED SDK already installed")
import pyzed.sl as sl

# --- Run depth pipeline ---
from compute_depth import (
    init_all_models, extract_svo_video,
    parse_robot_kinematics, align_temporal_streams, export_to_disk,
)
from core.depth import (
    compute_stereo_depth, build_universal_gripper_mask,
    distill_empirical_gripper_depth, inject_gripper_depth,
)

# Init models (only first time)
if 's2m2_model' not in dir():
    s2m2_model, run_stereo_matching, sam_predictor = init_all_models()

scene_constants = extract_svo_video(scene_constants)
scene_constants = parse_robot_kinematics(scene_constants)
scene_constants = align_temporal_streams(scene_constants)
scene_constants = compute_stereo_depth(
    scene_constants, s2m2_model, run_stereo_matching, device)

# Gripper refinement
wrist_serial = scene_constants["meta"].get("wrist_serial")
if wrist_serial and wrist_serial in scene_constants["camera"]:
    wrist_data = scene_constants["camera"][wrist_serial]
    if "raw_depth" in wrist_data:
        wrist_data["original_raw_depth"] = wrist_data["raw_depth"].copy()
scene_constants = build_universal_gripper_mask(scene_constants, sam_predictor)
scene_constants = distill_empirical_gripper_depth(scene_constants)
scene_constants = inject_gripper_depth(scene_constants)

export_to_disk(scene_constants)
print("✅ Stage 1 COMPUTE complete")

In [ ]:
# @title 1B. ☁️ LOAD depth from GCS bucket (skip Stage 1 computation)
# Run this if depth was already computed by run_parallel.sh.

GCS_DEPTH = "gs://dm-tapnet/mv-tap/droid/depth"
local_cache = f"/content/droid_depth_cache/{episode_id}"
os.makedirs(local_cache, exist_ok=True)

# Download robot data
os.system(f"gsutil cp '{GCS_DEPTH}/{episode_id}/robot.npz' '{local_cache}/' > /dev/null 2>&1")
robot_data = np.load(f"{local_cache}/robot.npz", allow_pickle=True)
for k in ['joint_positions', 'gripper_positions', 'T_cam_ee_init', 'T_ee_base_all']:
    if k in robot_data:
        scene_constants['robot'][k] = robot_data[k]
if 'valid_indices' in robot_data:
    scene_constants['meta']['valid_indices'] = robot_data['valid_indices']
if 'wrist_serial' in robot_data:
    scene_constants['meta']['wrist_serial'] = str(robot_data['wrist_serial'].item())
wrist_serial = scene_constants['meta'].get('wrist_serial')
print(f"  ✅ robot.npz loaded")

# Download per-camera data
base_files = ["video_left.mp4", "video_right.mp4",
              "video_left_raw.mp4", "video_right_raw.mp4",
              "raw_depth.npz", "calibration.npz"]

for cam in scene_constants['camera']:
    cam_dir = os.path.join(local_cache, cam)
    os.makedirs(cam_dir, exist_ok=True)

    cam_files = list(base_files)
    if cam == wrist_serial:
        cam_files.extend(["original_raw_depth.npz", "gripper_mask.npz", "gripper_depth.npz"])

    gcs_files = " ".join([f"'{GCS_DEPTH}/{episode_id}/{cam}/{f}'" for f in cam_files])
    os.system(f"gsutil -m cp {gcs_files} '{cam_dir}/' > /dev/null 2>&1")

    # Videos
    for mem_key, fname in [("video_rgb", "video_left.mp4"), ("video_right", "video_right.mp4"),
                           ("video_raw_rgb", "video_left_raw.mp4"), ("video_raw_right", "video_right_raw.mp4")]:
        vid_path = os.path.join(cam_dir, fname)
        if os.path.exists(vid_path):
            scene_constants['camera'][cam][mem_key] = media.read_video(vid_path)

    # Depth
    depth_path = os.path.join(cam_dir, "raw_depth.npz")
    if os.path.exists(depth_path):
        scene_constants['camera'][cam]['raw_depth'] = np.load(depth_path)['depth'].astype(np.float32) / 1000.0

    # Wrist extras
    for npz_key, mem_key, is_depth in [
        ("original_raw_depth.npz", "original_raw_depth", True),
        ("gripper_mask.npz", "sam_real_masks", False),
        ("gripper_depth.npz", "empirical_gripper_depth", True)]:
        p = os.path.join(cam_dir, npz_key)
        if os.path.exists(p):
            d = np.load(p)
            key = 'depth' if 'depth' in d else 'mask'
            val = d[key]
            if is_depth:
                val = val.astype(np.float32) / 1000.0
            scene_constants['camera'][cam][mem_key] = val

    # Calibration
    calib_path = os.path.join(cam_dir, "calibration.npz")
    if os.path.exists(calib_path):
        c = np.load(calib_path)
        scene_constants['camera'][cam]['K_mat'] = c['K_calib_left']
        scene_constants['camera'][cam]['baseline'] = float(c['baseline'])
        scene_constants['camera'][cam]['zed_calibration'] = {
            'calibrated': {'K': c['K_calib_left'], 'disto': c['disto_calib_left'],
                           'K_right': c['K_calib_right'], 'disto_right': c['disto_calib_right']},
            'raw': {'K': c['K_raw_left'], 'disto': c['disto_raw_left'],
                    'K_right': c['K_raw_right'], 'disto_right': c['disto_raw_right']},
        }
    print(f"  ✅ Camera {cam} loaded")

print(f"✅ Stage 1 LOADED from GCS")

In [ ]:
# @title 1. Visualize depth results
from utils.visualization import inspect_dict_structure, render_multicam_disparity_video

inspect_dict_structure(scene_constants)

frames = render_multicam_disparity_video(scene_constants, max_frames=30)
media.show_video(frames, fps=10, title="Depth [left | right | disparity] per camera")

---
## 1.5 Per-View 2D Point Tracking

Run **CoTracker** and **TAPNext** on all cameras.
Populates `tracking_results` for downstream track-reprojection metrics in Stage 2.

In [ ]:
# @title 1.5a. Run 2D tracking (CoTracker + TAPNext)
import importlib, compute_2d_tracks
importlib.reload(compute_2d_tracks)
from compute_2d_tracks import init_tracker, run_2d_tracking

GRID_SIZE = 30  # @param {type:"integer"}

tracking_results = {}
for method in ("cotracker", "tapnext"):
    _cache_key = f"_tracker_{method}"
    if _cache_key not in dir():
        globals()[_cache_key] = init_tracker(method, device)
    scene_constants = run_2d_tracking(
        globals()[_cache_key], scene_constants, device, grid_size=GRID_SIZE)
    tracking_results[method] = {
        cam_id: {
            'tracks_2d': scene_constants['camera'][cam_id]['tracks_2d'].copy(),
            'vis_2d': scene_constants['camera'][cam_id]['vis_2d'].copy(),
        }
        for cam_id in scene_constants['camera']
        if 'tracks_2d' in scene_constants['camera'][cam_id]
    }

print(f"✅ tracking_results ready: {list(tracking_results.keys())}")

In [ ]:
# @title 1.5b. Tracking video (select method)
import importlib, utils.visualization
importlib.reload(utils.visualization)
from utils.visualization import render_2d_tracking_video

VIS_METHOD = "cotracker"  # @param ["cotracker", "tapnext"]

if VIS_METHOD not in tracking_results:
    print(f"⚠️ Method '{VIS_METHOD}' not in tracking_results. "
          f"Available: {list(tracking_results.keys())}")
else:
    all_frames = []
    for cam_id in scene_constants['camera']:
        cam_data = scene_constants['camera'][cam_id]
        if cam_id not in tracking_results[VIS_METHOD]:
            continue
        tracks = tracking_results[VIS_METHOD][cam_id]['tracks_2d']
        vis = tracking_results[VIS_METHOD][cam_id]['vis_2d']
        frames = render_2d_tracking_video(
            cam_data['video_rgb'], tracks, vis,
            tgt_size=(256, 456), linewidth=1, max_frames=30)
        all_frames.append(np.array(frames))

    if all_frames:
        combined = np.concatenate(all_frames, axis=2)
        media.show_video(combined, fps=10,
                         title=f"2D Tracks [{VIS_METHOD}] — All Cameras")

---
## 2. Stage 2: Extrinsics

⚠️ **Run ONE of the two cells below** (Compute OR Load).

In [ ]:
# @title 2A. 🚧 COMPUTE extrinsics from scratch (VGGT + robot alignment)

from compute_extrinsics import (
    init_extrinsics,
    run_stage2_alignment, run_global_joint_alignment,
    evaluate_extrinsics, print_metrics, prepare_track_anchors,
    export_extrinsics,
)
from core.physics import PyBulletRenderer

# Init renderers (only first time)
if 'tensor_renderer' not in dir():
    from core.physics import TensorRobotRenderer
    tensor_renderer = TensorRobotRenderer(device=device)
if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()

def _eval_and_print(scene_state, stage_name):
    """Evaluate metrics + dual-tracker track reproj in one shot."""
    base = evaluate_extrinsics(scene_constants, scene_state, device,
                               pb_renderer=pb_renderer)
    print_metrics(base, stage_name)
    for method, tr in tracking_results.items():
        try:
            for cid, d in tr.items():
                scene_constants['camera'][cid]['tracks_2d'] = d['tracks_2d']
                scene_constants['camera'][cid]['vis_2d'] = d['vis_2d']
            anchors = prepare_track_anchors(
                scene_constants, scene_state, pb_renderer, device)
            m = evaluate_extrinsics(scene_constants, scene_state, device,
                                    pb_renderer=pb_renderer,
                                    track_anchors=anchors)
            wbg = m.get('track_reproj_wrist_bg_mean_px', float('nan'))
            wbg_med = m.get('track_reproj_wrist_bg_median_px', float('nan'))
            rob = m.get('track_reproj_static_robot_mean_px', float('nan'))
            rob_med = m.get('track_reproj_static_robot_median_px', float('nan'))
            parts = []
            if not np.isnan(wbg):
                parts.append(f"wrist_bg={wbg:.2f}/{wbg_med:.2f}")
            if not np.isnan(rob):
                parts.append(f"robot_fk={rob:.2f}/{rob_med:.2f}")
            print(f"  🎯 Track ({method:>10s}): {' | '.join(parts)} px (mean/median)")
        except Exception as e:
            print(f"  ⚠️ Track reproj ({method:>10s}): skipped — {e}")

# Stage 0+1: Dataset extrinsics → VGGT fallback
_vggt = globals().get('_vggt_models', None)
scene_state, _vggt_models = init_extrinsics(
    scene_constants, extrinsics_db, device, vggt_models=_vggt)
_eval_and_print(scene_state, "Stage 0+1 (Init)")

# Stage 2: Unified camera-robot alignment
scene_state = run_stage2_alignment(
    scene_constants, tensor_renderer, scene_state)
_eval_and_print(scene_state, "Stage 2 (Per-Camera)")

# Stage 3: Global joint optimization
scene_state = run_global_joint_alignment(
    scene_constants, scene_state, tensor_renderer)
_eval_and_print(scene_state, "Stage 3 (Global Joint)")

export_extrinsics(scene_constants, scene_state)
print("✅ Stage 2 COMPUTE complete")

In [ ]:
# @title 🔍 Visualize Extrinsics Metrics (all 6 items)
# Generates per-metric plots:
#   1. Robot Depth Loss — per-frame depth error for each camera
#   2. Chamfer Distance — per-frame 3D point cloud consistency
#   3. Track Wrist BG — per-frame FK reproj vs tracker (wrist, background pts)
#   4. Track Static Robot FK — per-frame FK 2d vs tracker 2d (static cams, robot pts)
#   5. Summary bar chart — all metrics side by side

import importlib, copy, torch, numpy as np, matplotlib.pyplot as plt
import compute_extrinsics; importlib.reload(compute_extrinsics)
import compute_2d_tracks; importlib.reload(compute_2d_tracks)
from compute_extrinsics import (prepare_track_anchors, compute_track_reproj_loss,
                                 evaluate_extrinsics, print_metrics)
from compute_2d_tracks import init_tracker, run_2d_tracking
from core.physics import PyBulletRenderer
from core.tracking import URDFKinematicsTracker
from core.pybullet_extrinsics import (
    get_foreground_robot_points, get_foreground_gripper_points,
    compute_robot_loss_batched, compute_wrist_loss_batched,
)

if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()

wrist_cam = scene_constants['meta']['wrist_serial']
ext_cams = [c for c in scene_constants['camera'].keys() if c != wrist_cam]
n_frames = len(scene_constants['robot']['joint_positions'])
T_ee_all = scene_constants['robot']['T_ee_base_all']

# ============================================================
# 1. ROBOT DEPTH LOSS — per-frame depth error per camera
# ============================================================
print("=" * 70)
print("📊 1. Robot Depth Loss (PyBullet rendered vs sensor depth)")
print("=" * 70)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
all_cam_ids = list(ext_cams) + [wrist_cam]
labels = ['cam1', 'cam2', 'wrist']

for ax, cam_id, label in zip(axes, all_cam_ids, labels):
    is_wrist = (cam_id == wrist_cam)
    K_np = scene_constants['camera'][cam_id]['K_mat']
    per_frame_err = []

    for t in range(n_frames):
        joints = scene_constants['robot']['joint_positions'][t]
        gripper = scene_constants['robot']['gripper_positions'][t]
        pb_renderer.update_robot_pose(joints, gripper_state=gripper)

        ext_t = scene_state[cam_id]['extrinsics'][t]
        d_obs = scene_constants['camera'][cam_id]['raw_depth'][t].astype(np.float32)
        h_img, w_img = d_obs.shape
        d_render = pb_renderer.render_depth(ext_t, K_np, w_img, h_img)

        valid = (d_render > 0.01) & (d_render < 1.5) & (d_obs > 0.01) & (d_obs < 1.5)
        if valid.any():
            per_frame_err.append(np.abs(d_render[valid] - d_obs[valid]).mean())
        else:
            per_frame_err.append(np.nan)

    per_frame_err = np.array(per_frame_err)
    ax.plot(per_frame_err, linewidth=0.8)
    ax.axhline(y=np.nanmean(per_frame_err), color='r', linestyle='--',
               label=f'mean={np.nanmean(per_frame_err):.4f}m')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Mean |Δ depth| (m)')
    ax.set_title(f'{label} [{cam_id[:8]}]')
    ax.legend(fontsize=8)
plt.suptitle('Robot Depth Loss per Frame', fontsize=14)
plt.tight_layout()
plt.show()

# ============================================================
# 2. CHAMFER DISTANCE — per-frame 3D point cloud consistency
# ============================================================
print("\n" + "=" * 70)
print("📊 2. Chamfer Distance (3D point cloud consistency)")
print("=" * 70)

from compute_extrinsics import get_cam_points_local_t, batched_chamfer_distance

cam1, cam2 = ext_cams[0], ext_cams[1]
T1 = torch.tensor(scene_state[cam1]['base_extrinsic'], dtype=torch.float32, device=device)
T2 = torch.tensor(scene_state[cam2]['base_extrinsic'], dtype=torch.float32, device=device)
Tw = torch.tensor(scene_state[wrist_cam]['base_extrinsic'], dtype=torch.float32, device=device)

chamfer_12, chamfer_1w, chamfer_2w = [], [], []
for t in range(n_frames):
    pc1 = get_cam_points_local_t(t, scene_constants['camera'][cam1], device)
    pc2 = get_cam_points_local_t(t, scene_constants['camera'][cam2], device)
    pcw = get_cam_points_local_t(t, scene_constants['camera'][wrist_cam], device)
    if pc1 is None or pc2 is None or pcw is None:
        chamfer_12.append(np.nan); chamfer_1w.append(np.nan); chamfer_2w.append(np.nan)
        continue
    Tee_t = torch.tensor(T_ee_all[t], dtype=torch.float32, device=device)
    w1 = (T1 @ pc1)[:3, :].T.unsqueeze(0)
    w2 = (T2 @ pc2)[:3, :].T.unsqueeze(0)
    ww = ((Tee_t @ Tw) @ pcw)[:3, :].T.unsqueeze(0)
    l12, _ = batched_chamfer_distance(w1, w2, device)
    l1w, _ = batched_chamfer_distance(w1, ww, device)
    l2w, _ = batched_chamfer_distance(w2, ww, device)
    chamfer_12.append(l12.item())
    chamfer_1w.append(l1w.item())
    chamfer_2w.append(l2w.item())

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.plot(chamfer_12, label=f'cam1↔cam2 (mean={np.nanmean(chamfer_12):.4f})', linewidth=0.8)
ax.plot(chamfer_1w, label=f'cam1↔wrist (mean={np.nanmean(chamfer_1w):.4f})', linewidth=0.8)
ax.plot(chamfer_2w, label=f'cam2↔wrist (mean={np.nanmean(chamfer_2w):.4f})', linewidth=0.8)
ax.set_xlabel('Frame')
ax.set_ylabel('Chamfer Distance (m)')
ax.set_title('Chamfer Distance per Frame (lower = better alignment)')
ax.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3. TRACK WRIST BG — wrist background reproj per frame
# ============================================================
print("\n" + "=" * 70)
print("📊 3. Track Wrist Background (FK reproj vs 2D tracker)")
print("=" * 70)

tracker_name = "tapnext"   # change to "cotracker" if desired
if 'dbg_tracker' not in dir() or dbg_tracker.name.lower() != tracker_name:
    dbg_tracker = init_tracker(tracker_name, device)

sc_dbg = copy.deepcopy(scene_constants)
sc_dbg = run_2d_tracking(dbg_tracker, sc_dbg, device, grid_size=30)
anchors = prepare_track_anchors(sc_dbg, scene_state, pb_renderer, device)
T_ee_t = torch.tensor(T_ee_all, dtype=torch.float32, device=device)

for cam_id, anchor in anchors.items():
    scheme = anchor['scheme']
    if scheme == 'robot_fk':
        # Already computed in prepare_track_anchors
        print(f"  [{cam_id}] robot_fk: mean={anchor['mean_px']:.2f} "
              f"median={anchor['median_px']:.2f} px (see section 4)")
        continue
    if scheme != 'background':
        continue

    T_opt = torch.tensor(scene_state[cam_id]['base_extrinsic'],
                         dtype=torch.float32, device=device)
    K_t = torch.tensor(sc_dbg['camera'][cam_id]['K_mat'],
                       dtype=torch.float32, device=device)

    P_cam0 = anchor['P_cam0']
    targets = anchor['tracks_2d']
    vis = anchor['vis']

    T_cam_to_world_0 = T_ee_t[0] @ T_opt
    P_world = T_cam_to_world_0 @ P_cam0.T
    T_cam_to_world_all = T_ee_t @ T_opt.unsqueeze(0)
    T_world_to_cam_all = torch.linalg.inv(T_cam_to_world_all)
    P_cam_all = T_world_to_cam_all @ P_world.unsqueeze(0)

    Z = P_cam_all[:, 2, :].clamp(min=1e-4)
    u_pred = K_t[0, 0] * P_cam_all[:, 0, :] / Z + K_t[0, 2]
    v_pred = K_t[1, 1] * P_cam_all[:, 1, :] / Z + K_t[1, 2]
    pred = torch.stack([u_pred, v_pred], dim=-1)

    pixel_err = (pred - targets).abs().sum(dim=-1)
    valid = vis & (Z > 0.05)
    if 'eval_frame_mask' in anchor:
        valid = valid & anchor['eval_frame_mask'][:, None]

    err_np = pixel_err.detach().cpu().numpy()
    valid_np = valid.detach().cpu().numpy()

    per_frame = [err_np[t, valid_np[t]].mean() if valid_np[t].any() else np.nan
                 for t in range(err_np.shape[0])]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    # Per-frame plot
    axes[0].plot(per_frame, linewidth=0.8)
    overall = np.nanmean(per_frame)
    axes[0].axhline(y=overall, color='r', linestyle='--', label=f'mean={overall:.1f}px')
    axes[0].set_xlabel('Frame')
    axes[0].set_ylabel('L1 px error')
    axes[0].set_title(f'[{cam_id[:8]}] Wrist BG per-frame')
    axes[0].legend()

    # Per-track histogram
    per_track = [err_np[valid_np[:, n], n].mean() if valid_np[:, n].any() else np.nan
                 for n in range(err_np.shape[1])]
    axes[1].hist([x for x in per_track if not np.isnan(x)], bins=30)
    axes[1].set_xlabel('Mean L1 px error')
    axes[1].set_title(f'[{cam_id[:8]}] Per-track distribution')

    # Heatmap
    im = axes[2].imshow(err_np * valid_np, aspect='auto', cmap='hot',
                         vmin=0, vmax=min(50, np.nanmax(err_np[valid_np])))
    axes[2].set_xlabel('Track idx')
    axes[2].set_ylabel('Frame')
    axes[2].set_title(f'[{cam_id[:8]}] Error heatmap')
    plt.colorbar(im, ax=axes[2], label='px')

    plt.suptitle(f'Wrist BG Track Reproj ({tracker_name})', fontsize=14)
    plt.tight_layout()
    plt.show()

# ============================================================
# 4. TRACK STATIC ROBOT FK — FK 2D vs tracker 2D on static cams
# ============================================================
print("\n" + "=" * 70)
print("📊 4. Track Static Robot (URDF FK vs 2D tracker)")
print("=" * 70)

for cam_id in ext_cams:
    cam_data = sc_dbg['camera'][cam_id]
    if 'tracks_2d' not in cam_data:
        print(f"  ⚠️ [{cam_id}] No tracks, skipping.")
        continue

    pb_renderer.update_robot_pose(
        scene_constants['robot']['joint_positions'][0],
        gripper_state=scene_constants['robot']['gripper_positions'][0])
    urdf_tracker = URDFKinematicsTracker(pb_renderer)
    result = urdf_tracker.extract_robot_tracks(cam_id, sc_dbg, scene_state)
    traj_3d, traj_2d_fk, vis_fk, robot_indices = result

    if robot_indices is None or len(robot_indices) < 5:
        print(f"  ⚠️ [{cam_id}] <5 robot points found.")
        continue

    tracker_2d = cam_data['tracks_2d'][:, robot_indices]
    tracker_vis = cam_data['vis_2d'][:, robot_indices]
    combined_vis = vis_fk & tracker_vis
    pixel_err = np.abs(traj_2d_fk - tracker_2d).sum(axis=-1)
    valid = combined_vis & (pixel_err < 500)

    per_frame = [pixel_err[t, valid[t]].mean() if valid[t].any() else np.nan
                 for t in range(pixel_err.shape[0])]
    per_track = [pixel_err[valid[:, n], n].mean() if valid[:, n].any() else np.nan
                 for n in range(pixel_err.shape[1])]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].plot(per_frame, linewidth=0.8)
    overall = np.nanmean(per_frame)
    axes[0].axhline(y=overall, color='r', linestyle='--', label=f'mean={overall:.1f}px')
    axes[0].set_xlabel('Frame')
    axes[0].set_ylabel('L1 px error')
    axes[0].set_title(f'[{cam_id[:8]}] Robot FK per-frame')
    axes[0].legend()

    axes[1].hist([x for x in per_track if not np.isnan(x)], bins=30)
    axes[1].set_xlabel('Mean L1 px error')
    axes[1].set_title(f'[{cam_id[:8]}] Per-track distribution')

    im = axes[2].imshow(pixel_err * valid, aspect='auto', cmap='hot',
                         vmin=0, vmax=min(50, np.nanmax(pixel_err[valid])))
    axes[2].set_xlabel('Track idx')
    axes[2].set_ylabel('Frame')
    axes[2].set_title(f'[{cam_id[:8]}] Error heatmap')
    plt.colorbar(im, ax=axes[2], label='px')

    plt.suptitle(f'Static Robot FK Track ({tracker_name}) [{cam_id[:8]}]', fontsize=14)
    plt.tight_layout()
    plt.show()

    # Top-5 worst tracks with frame 0 position
    per_track_arr = np.array(per_track)
    worst_idx = np.argsort(np.nan_to_num(per_track_arr, nan=-1))[::-1][:5]
    print(f"  Top-5 worst tracks [{cam_id[:8]}]:")
    for i, idx in enumerate(worst_idx):
        u0 = tracker_2d[0, idx, 0]
        v0 = tracker_2d[0, idx, 1]
        print(f"    #{i+1}: track {robot_indices[idx]} (u={u0:.0f}, v={v0:.0f}) "
              f"mean_err={per_track_arr[idx]:.1f} px")

# ============================================================
# 5. SUMMARY — all metrics bar chart
# ============================================================
print("\n" + "=" * 70)
print("📊 5. Summary Bar Chart")
print("=" * 70)

metrics = evaluate_extrinsics(sc_dbg, scene_state, device,
                              pb_renderer=pb_renderer,
                              track_anchors=anchors)
print_metrics(metrics, f"Summary ({tracker_name})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: depth-related metrics (meters)
depth_names = ['chamfer_total', 'robot_loss_cam1', 'robot_loss_cam2', 'robot_loss_wrist']
depth_vals = [metrics.get(k, 0) for k in depth_names]
depth_labels = ['Chamfer\ntotal', 'Robot\ncam1', 'Robot\ncam2', 'Robot\nwrist']
bars = axes[0].bar(depth_labels, depth_vals, color=['#2196F3', '#FF9800', '#FF9800', '#FF9800'])
for bar, val in zip(bars, depth_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)
axes[0].set_ylabel('Error (meters)')
axes[0].set_title('3D Metrics')

# Right: track-related metrics (pixels)
track_names = ['track_reproj_wrist_bg_mean_px', 'track_reproj_wrist_bg_median_px',
               'track_reproj_static_robot_mean_px', 'track_reproj_static_robot_median_px']
track_vals = [metrics.get(k, float('nan')) for k in track_names]
track_labels = ['Wrist BG\nmean', 'Wrist BG\nmedian', 'Robot FK\nmean', 'Robot FK\nmedian']
colors = ['#4CAF50', '#81C784', '#9C27B0', '#BA68C8']
bars2 = axes[1].bar(track_labels, [v if not np.isnan(v) else 0 for v in track_vals], color=colors)
for bar, val in zip(bars2, track_vals):
    label = f'{val:.1f}' if not np.isnan(val) else 'N/A'
    axes[1].text(bar.get_x() + bar.get_width()/2, max(bar.get_height(), 0.5),
                 label, ha='center', va='bottom', fontsize=9)
axes[1].set_ylabel('Error (pixels)')
axes[1].set_title(f'2D Track Metrics ({tracker_name})')

plt.suptitle(f'Extrinsics Quality Summary — {episode_id}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# @title 2B. ☁️ LOAD extrinsics from GCS bucket (skip extrinsics computation)

GCS_EXT = "gs://dm-tapnet/mv-tap/droid/extrinsics"
local_ext_cache = f"/content/droid_extrinsics_cache/{episode_id}"
os.makedirs(local_ext_cache, exist_ok=True)

scene_state = {}
for cam in scene_constants['camera']:
    cam_dir = os.path.join(local_ext_cache, cam)
    os.makedirs(cam_dir, exist_ok=True)

    gcs_path = f"{GCS_EXT}/{episode_id}/{cam}/extrinsics.json"
    local_path = os.path.join(cam_dir, "extrinsics.json")
    os.system(f"gsutil cp '{gcs_path}' '{local_path}' > /dev/null 2>&1")

    if os.path.exists(local_path):
        with open(local_path) as f:
            ext_data = json.load(f)
        scene_state[cam] = {
            'base_extrinsic': np.array(ext_data['base_extrinsic'], dtype=np.float32),
            'extrinsics': np.array(ext_data['extrinsics'], dtype=np.float32),
            'is_wrist': ext_data.get('is_wrist', False),
        }
        flag = "🦿" if scene_state[cam]['is_wrist'] else "🎥"
        print(f"  ✅ [{cam}] {flag} Shape: {scene_state[cam]['extrinsics'].shape}")
    else:
        print(f"  ⚠️ [{cam}] missing")

# --- Evaluate loaded extrinsics ---
from compute_extrinsics import evaluate_extrinsics, print_metrics, prepare_track_anchors
from core.physics import PyBulletRenderer
if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()

base = evaluate_extrinsics(scene_constants, scene_state, device,
                           pb_renderer=pb_renderer)
print_metrics(base, "Final Extrinsics (GCS)")
for method, tr in tracking_results.items():
    try:
        for cid, d in tr.items():
            scene_constants['camera'][cid]['tracks_2d'] = d['tracks_2d']
            scene_constants['camera'][cid]['vis_2d'] = d['vis_2d']
        anchors = prepare_track_anchors(
            scene_constants, scene_state, pb_renderer, device)
        m = evaluate_extrinsics(scene_constants, scene_state, device,
                                pb_renderer=pb_renderer,
                                track_anchors=anchors)
        wbg = m.get('track_reproj_wrist_bg_mean_px', float('nan'))
        wbg_med = m.get('track_reproj_wrist_bg_median_px', float('nan'))
        rob = m.get('track_reproj_static_robot_mean_px', float('nan'))
        rob_med = m.get('track_reproj_static_robot_median_px', float('nan'))
        parts = []
        if not np.isnan(wbg):
            parts.append(f"wrist_bg={wbg:.2f}/{wbg_med:.2f}")
        if not np.isnan(rob):
            parts.append(f"robot_fk={rob:.2f}/{rob_med:.2f}")
        print(f"  🎯 Track ({method:>10s}): {' | '.join(parts)} px (mean/median)")
    except Exception as e:
        print(f"  ⚠️ Track reproj ({method:>10s}): skipped — {e}")

print("✅ Extrinsics LOADED from GCS")

In [ ]:
# @title 2a. Camera axes overlay
from utils.visualization import render_cross_camera_axes

try:
    axes_frames = render_cross_camera_axes(scene_constants, scene_state, max_frames=30)
    if axes_frames:
        media.show_video(axes_frames, fps=10, title="Camera Axes Overlay")
except Exception as e:
    print(f"Axes visualization skipped: {e}")

In [ ]:
# @title 2b. Robot segmentation video
from utils.visualization import render_segmentation_video
from core.physics import PyBulletRenderer

if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()

try:
    seg_frames = render_segmentation_video(scene_constants, scene_state, pb_renderer, max_frames=30)
    if seg_frames:
        media.show_video(seg_frames, fps=10, title="Robot Mask Overlay")
except Exception as e:
    print(f"Segmentation visualization skipped: {e}")

In [ ]:
# @title 2c. Fused 3D point cloud
from utils.visualization import render_fused_point_cloud

try:
    render_fused_point_cloud(scene_constants, scene_state, frame_idx=0, height=600, width=1000)
except Exception as e:
    print(f"Fused point cloud skipped: {e}")

In [ ]:
# @title 2d. 4D cinematic orbit
from utils.visualization import render_cinematic_4d_orbit

try:
    orbit_frames = render_cinematic_4d_orbit(scene_constants, scene_state, max_frames=30)
    media.show_video(orbit_frames, fps=10, title="4D Orbit")
except Exception as e:
    print(f"4D Orbit skipped: {e}")

---
## 3. Stage 3: Tracking

⚠️ **Run ONE of the two cells below** (Compute OR Load).

In [ ]:
# @title 3A. 🚧 COMPUTE tracks from scratch (CoTracker + URDF + multi-view fusion)

from compute_tracks import (
    init_tracking_models,
    phase1_extract_2d_tracks, phase2_lift_and_filter,
    phase3_3d_dedup, phase4_cross_view_completion,
    phase5_median_3d_fusion, export_tracks,
)

# Init models (only first time)
if 'cotracker_model' not in dir():
    cotracker_model = init_tracking_models()

camera_ids = list(scene_constants['camera'].keys())

# Phase 1-5
scene_constants = phase1_extract_2d_tracks(cotracker_model, scene_constants, device)
per_cam_env = phase2_lift_and_filter(scene_constants, scene_state, pb_renderer)
unified_pts_3d, unified_to_cam, N_unified = phase3_3d_dedup(per_cam_env, camera_ids)
per_cam_tracks, per_cam_vis = phase4_cross_view_completion(
    cotracker_model, scene_constants, scene_state,
    per_cam_env, unified_pts_3d, unified_to_cam, N_unified, device)
(final_traj_3d, final_vis_global,
 final_per_cam_tracks, final_per_cam_vis, n_survived) = phase5_median_3d_fusion(
    scene_constants, scene_state, per_cam_tracks, per_cam_vis, N_unified)

export_tracks(scene_constants, scene_state,
              final_traj_3d, final_vis_global,
              final_per_cam_tracks, final_per_cam_vis)
print(f"✅ Stage 3 COMPUTE complete: {final_traj_3d.shape[1]} points")

In [ ]:
# @title 3B. ☁️ LOAD tracks from GCS bucket (skip Stage 3 computation)

GCS_TRACKS = "gs://dm-tapnet/mv-tap/droid/tracks"
local_tracks_cache = f"/content/droid_tracks_cache/{episode_id}"
os.makedirs(local_tracks_cache, exist_ok=True)

# Download global 3D tracks + metadata
for fname in ["tracks_3d.npz", "track_metadata.npz"]:
    gcs_path = f"{GCS_TRACKS}/{episode_id}/{fname}"
    local_path = os.path.join(local_tracks_cache, fname)
    ret = os.system(f"gsutil cp '{gcs_path}' '{local_path}' > /dev/null 2>&1")
    if ret == 0:
        print(f"  ✅ {fname}")
    else:
        print(f"  ⚠️  {fname} not found (skipping)")

# Load global tracks
data_3d = np.load(os.path.join(local_tracks_cache, "tracks_3d.npz"))
final_traj_3d = data_3d["traj_3d"]       # (T, N, 3)
final_vis_global = data_3d["vis_global"]  # (T, N)

# Load env/robot split metadata
meta_path = os.path.join(local_tracks_cache, "track_metadata.npz")
if os.path.exists(meta_path):
    meta = np.load(meta_path)
    n_env   = int(meta["n_env"])
    n_robot = int(meta["n_robot"])
else:
    n_env, n_robot = final_traj_3d.shape[1], 0

T, N, _ = final_traj_3d.shape
print(f"  ✅ tracks_3d: {T} frames × {N} points ({n_env} env + {n_robot} robot)")

# Download per-camera 2D tracks
final_per_cam_tracks = {}
final_per_cam_vis = {}

for cam_id in scene_constants["camera"]:
    cam_cache = os.path.join(local_tracks_cache, cam_id)
    os.makedirs(cam_cache, exist_ok=True)

    gcs_cam = f"{GCS_TRACKS}/{episode_id}/{cam_id}"
    for fname in ["tracks_2d.npz", "intrinsics.npy", "extrinsics_w2c.npy"]:
        os.system(f"gsutil cp '{gcs_cam}/{fname}' '{cam_cache}/' > /dev/null 2>&1")

    t2d_path = os.path.join(cam_cache, "tracks_2d.npz")
    if os.path.exists(t2d_path):
        d = np.load(t2d_path)
        final_per_cam_tracks[cam_id] = d["traj_2d"]   # (T, N, 2)
        final_per_cam_vis[cam_id]    = d["vis_2d"]     # (T, N)
        print(f"  ✅ Camera [{cam_id}]: 2D tracks loaded")
    else:
        print(f"  ⚠️  Camera [{cam_id}]: tracks_2d.npz not found")

print(f"\n✅ Stage 3 LOADED from GCS — {len(final_per_cam_tracks)} cameras")

In [ ]:
# @title 3. Visualize tracking results
from utils.visualization import render_2d_tracking_video

camera_ids = list(scene_constants['camera'].keys())

all_frames = []
for cam_id in camera_ids:
    cam_data = scene_constants['camera'][cam_id]
    if cam_id in final_per_cam_tracks:
        tracks = final_per_cam_tracks[cam_id]
        vis = final_per_cam_vis[cam_id]
        frames = render_2d_tracking_video(
            cam_data['video_rgb'], tracks, vis,
            tgt_size=(256, 456), linewidth=1, max_frames=30)
        all_frames.append(np.array(frames))

if all_frames:
    combined = np.concatenate(all_frames, axis=2)  # horizontal concat
    media.show_video(combined, fps=10, title="Tracks [all cameras]")

---
## Summary

```
per stage:
  A. 🚧 COMPUTE — run from scratch (debug single episode)
  B. ☁️ LOAD    — pull from GCS   (skip to later stages)

droid/
├── compute_depth.py          → Stage 1
├── compute_extrinsics.py     → Stage 2
├── compute_tracks.py         → Stage 3
├── core/                     → Shared modules
│   ├── geometry.py, io.py, depth.py, physics.py, tracking.py
└── utils/visualization.py    → All viz helpers
```

Typical debug workflow:
1. Run Stage 1 once → `run_parallel.sh` on GCP → results on GCS
2. Open notebook → **1B. Load** depth from GCS → 🚧 debug Stage 2
3. Stage 2 works → `run_parallel.sh --stage 2` → results on GCS
4. Open notebook → **1B + 2B. Load** both → 🚧 debug Stage 3